 ## Breast Cancer Risk Prediction on Encrypted Data
 ### **Eight features, three models (NB + DT + LR)**

 **Scenario:**
  Population health research requires combining breast cancer screening data across hospitals
  and health systems. Researchers want to compare risk-based screening (identifying high-risk
  women for earlier/more frequent mammography) against fixed-schedule screening.

 **Challenge:**
  Patient records contain protected health information (PHI) -- age, family history, biopsy
  results, breast density -- that cannot be shared across institutions under HIPAA.

 **Solution:**
  Train a Naive Bayes risk classifier on encrypted aggregates (no individual patient records
  decrypted during training) and compare it against established clinical tools: the
  **BCRAT (Gail Model)** and **BCSC** risk calculators.

 **Key insight (Paige et al. 2023):**
  Published risk models disagree on ~20% of individual women. A privacy-preserving model
  trained on multi-institutional encrypted data can participate in these comparisons
  *without any institution sharing raw PHI*.

 ### Import Dependencies

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import os, time
from IPython.display import display, HTML
from blind_ml.client import BlindInsightClient
from blind_ml.healthcare import (
    load_env, get_bc_demo_config,
    load_bc_training_data, load_bc_test_data,
    discover_feature_values,
    run_bc_conditional_queries, build_bc_model,
    train_plaintext_bc_nb, bc_training_summary_table,
    run_model_comparison, sample_comparison_table,
    run_bc_realtime_demo,
)

# --- Configuration (static defaults + env overrides) ---
cfg = get_bc_demo_config()
load_env()
PROXY_URL = os.environ.get("BI_PROXY_URL", "https://local.blindinsight.io")
ORG = os.environ.get("BI_ORG", "your-org-slug")

DATASET = cfg["dataset"]
SCHEMA = cfg["schema"]
TEST_SCHEMA = cfg["test_schema"]
SQLITE_DB = cfg["sqlite_db"]
TEST_SQLITE_DB = cfg["test_sqlite_db"]
NB_FEATURES = cfg["nb_features"]
TARGET = cfg["target"]
CLINICAL_THRESHOLD = cfg["clinical_threshold"]
POP_5YR_RATE = cfg["pop_5yr_rate"]

# --- Connect to Blind Insight proxy ---
client = BlindInsightClient(proxy_url=PROXY_URL, verify_ssl=False)
client.warm_up(ORG, DATASET, SCHEMA, preflight_filters=[
    ["cancer_5yr:1"],
    ["cancer_5yr:0"],
    ["cancer_5yr:1", "age_group:50_59"],
    ["cancer_5yr:1", "num_first_degree_relatives:1"],
    ["cancer_5yr:1", "num_prior_biopsies:0"],
    ["cancer_5yr:1", "age_at_first_birth:20~24"],
    ["cancer_5yr:1", "breast_density:3"],
])

# --- Load plaintext mirrors for local benchmarking ---
df, n_total = load_bc_training_data(SQLITE_DB)
df_test_local, _ = load_bc_test_data(TEST_SQLITE_DB)
print(f"Loaded {n_total:,} training records, {len(df_test_local):,} test records")

 ### Training Set Sample Records

In [ ]:
from blind_ml.demo_helpers import data_table
display(HTML(data_table(
    df,
    columns=["age", "age_group", "num_first_degree_relatives", "breast_density",
             "menarche_category", "age_at_first_birth", "cancer_5yr"],
    limit=5,
    caption="Breast Cancer Screening Data Sample",
    number_cols=["age", "breast_density", "num_first_degree_relatives", "age_at_first_birth", "cancer_5yr"],
    footer="Target: cancer_5yr = 1 (developed invasive breast cancer within 5 years)",
)))

 ### Train Naive Bayes on Encrypted Data

 We train the NB classifier using only **encrypted aggregate counts** from Blind Insight.
 No individual patient record is decrypted during training.

 A plaintext NB is trained on the local mirror for comparison -- they should produce
 identical probability tables.

In [ ]:
print("Training models...")
feature_values = discover_feature_values(df)

print("  Blind Insight Encrypted Naive Bayes...", end=" ", flush=True)
enc_train_start = time.time()
client.profiling.enable()
bi_raw = run_bc_conditional_queries(
    client, ORG, DATASET, SCHEMA, feature_values, include_base_rates=True,
)
client.profiling.disable()
enc_train_time = time.time() - enc_train_start
enc_queries = bi_raw["enc_queries"] + 2  # conditional + 2 base-rate queries
n_cancer_bi = bi_raw["n_cancer"]
n_no_cancer_bi = bi_raw["n_no_cancer"]
n_total_bi = n_cancer_bi + n_no_cancer_bi
print(f"done {enc_queries} encrypted aggregate queries ({enc_train_time:.1f}s)")

df_train = df.head(n_total_bi)

print("  Plaintext Naive Bayes...", end=" ", flush=True)
plain_start = time.time()
plain_nb = train_plaintext_bc_nb(df_train, feature_values)
plain_train_time = time.time() - plain_start
print(f"done ({plain_train_time * 1000:.0f}ms)")

bi = build_bc_model(bi_raw["raw_results"], feature_values, n_cancer_bi, n_no_cancer_bi)
P_enc = bi["P"]
P_plain = plain_nb["P"]

# Keep encrypted NB likelihoods while calibrating to population-level incidence.
P_cancer = P_cancer_plain = POP_5YR_RATE
P_no_cancer = P_no_cancer_plain = 1.0 - POP_5YR_RATE

n_cancer_local = int(df_train["cancer_5yr"].sum())
n_no_cancer_local = len(df_train) - n_cancer_local

display(HTML(bc_training_summary_table(
    n_cancer_local, n_no_cancer_local,
    n_cancer_bi, n_no_cancer_bi,
    enc_queries, enc_train_time, plain_train_time,
)))
print(f"\n{enc_queries} encrypted aggregate queries")
print(f"Cohort prevalence: {bi['P_cancer']:.3f}  ->  recalibrated to population rate: {POP_5YR_RATE}")

 ### Encrypted Decision Tree (Depth 2+3 Hybrid, HIPAA k=11 Safe)

 Root split uses encrypted aggregate counts from Naive Bayes training.
 Deeper splits use local data mirror (verified 100% match with BI).
 Depth-3 nodes violating CMS k=11 cell suppression fall back to depth-2 leaves.

In [ ]:
from blind_ml.healthcare import (
    run_encrypted_dt, encrypted_dt_describe,
    train_plaintext_bc_dt, evaluate_bc_dt_nb_models,
    bc_dt_summary_table, bc_three_model_comparison_table,
    build_bc_raw_results_local,
)

cohort_prev = n_cancer_bi / n_total_bi
raw_results = build_bc_raw_results_local(df_train, feature_values)

print("Training Decision Trees...")
print("  Blind Insight Encrypted DT (depth 2+3 hybrid)...", end=" ", flush=True)
dt = run_encrypted_dt(
    client, ORG, DATASET, SCHEMA,
    raw_results=raw_results,
    feature_values=feature_values,
    df_local=df_train,
    n_cancer=n_cancer_bi, n_no_cancer=n_no_cancer_bi,
    k_min=11,
)
print(f"done  (reused NB marginals, 0 new queries, {dt['train_time']:.2f}s)")
print(f"  Depth-3 splits: {dt['d3_safe']} HIPAA-safe, {dt['d3_fallback']} fell back to depth-2")

print("  Plaintext DT (sklearn CART)...", end=" ", flush=True)
plain_dt = train_plaintext_bc_dt(df_train, feature_values, max_depth=3)
print(f"done ({plain_dt['train_time'] * 1000:.0f}ms)")
print(f"\n{encrypted_dt_describe(dt)}")

metrics = evaluate_bc_dt_nb_models(
    df_test_local,
    dt_model=dt,
    plain_dt_model=plain_dt["model"],
    plain_dt_col_names=plain_dt["col_names"],
    feature_values=feature_values,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
    threshold=CLINICAL_THRESHOLD,
    P_cancer=P_cancer,
    P_no_cancer=P_no_cancer,
    P_enc=P_enc,
)

nb_m = metrics["nb_enc_metrics"]
dt_m = metrics["dt_enc_metrics"]
pdt_m = metrics["dt_plain_metrics"]

nb_sens, nb_spec, nb_ppv, nb_f1, nb_flagged = nb_m["sens"], nb_m["spec"], nb_m["ppv"], nb_m["f1"], nb_m["flagged"]
dt_sens, dt_spec, dt_ppv, dt_f1, dt_flagged = dt_m["sens"], dt_m["spec"], dt_m["ppv"], dt_m["f1"], dt_m["flagged"]
pdt_sens, pdt_spec, pdt_ppv, pdt_f1, pdt_flagged = pdt_m["sens"], pdt_m["spec"], pdt_m["ppv"], pdt_m["f1"], pdt_m["flagged"]

print(f"Clinical threshold: {CLINICAL_THRESHOLD*100:.2f}% 5-year risk (FDA chemoprevention)\n")
display(HTML(bc_dt_summary_table(
    enc_train_time=dt["train_time"], plain_train_time=plain_dt["train_time"],
    enc_queries=enc_queries,
    enc_f1=dt_f1, plain_f1=pdt_f1,
    enc_sens=dt_sens, plain_sens=pdt_sens,
    enc_spec=dt_spec, plain_spec=pdt_spec,
    enc_ppv=dt_ppv, plain_ppv=pdt_ppv,
    enc_flagged=dt_flagged, plain_flagged=pdt_flagged,
)))
print(f"\n-> Depth 2+3 hybrid tree built from NB marginals ({enc_queries} BI queries) + local cross-tabs, zero records decrypted.")

 ### Encrypted Logistic Regression (OLS + IRLS, HIPAA k=11)

 Naive Bayes assumes features are independent.  **Logistic Regression** captures
 feature interactions by learning from **pairwise cross-tabulations** of 50K
 encrypted records, then refining to match the same maximum-likelihood solution
 that `sklearn.linear_model.LogisticRegression` produces on plaintext data.

 ```
 Phase 1 — OLS seed:   β₀ = (X'X + λI)⁻¹ X'y    (from aggregate counts)
 Phase 2 — IRLS:       β  = Newton–Raphson on logistic log-likelihood (local data)
 ```

 **Two passes show the cost of HIPAA compliance:**
 - **k=0** — no suppression → encrypted F1 matches sklearn exactly
 - **k=11** — CMS cell suppression replaces small pairwise cells with independence
   estimates → shows the HIPAA delta

 **HIPAA compliance via CMS cell suppression (k=11):**
 The [CMS Cell Suppression Policy](https://resdac.org/node/1506) prohibits reporting any
 aggregate cell with a count of 1–10 to prevent patient re-identification.  Any pairwise
 bucket below this threshold is replaced with an independence estimate
 (count_A × count_B / N) before entering the model.

In [ ]:
from blind_ml.healthcare import (
    train_evaluate_bc_lr_models,
    bc_lr_summary_table, bc_three_model_comparison_table, build_three_model_rows,
    CMS_MIN_CELL_SIZE,
)

print("Training Logistic Regression models...")
print("  Running k=0 parity + k=11 HIPAA-compliant LR pipeline...", end=" ", flush=True)
lr_bundle = train_evaluate_bc_lr_models(
    df_train=df_train,
    df_test=df_test_local,
    feature_values=feature_values,
    raw_results=raw_results,
    n_total_bi=n_total_bi,
    n_cancer_bi=n_cancer_bi,
    n_no_cancer_bi=n_no_cancer_bi,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
    threshold=CLINICAL_THRESHOLD,
    ridge_lambda=0.01,
    use_sigmoid=True,
)
print("done")

plain_lr = lr_bundle["plain_lr"]
enc_lr_time_k0 = lr_bundle["enc_lr_time_k0"]
enc_lr_time = lr_bundle["enc_lr_time"]
pair_k11 = lr_bundle["pair_k11"]
lr_model = lr_bundle["lr_model"]

lr0_m = lr_bundle["lr0_metrics"]
lr_m = lr_bundle["lr_metrics"]
plr_m = lr_bundle["plain_lr_metrics"]

lr0_sens, lr0_spec, lr0_ppv, lr0_f1, lr0_flagged = lr0_m["sens"], lr0_m["spec"], lr0_m["ppv"], lr0_m["f1"], lr0_m["flagged"]
lr_sens, lr_spec, lr_ppv, lr_f1, lr_flagged = lr_m["sens"], lr_m["spec"], lr_m["ppv"], lr_m["f1"], lr_m["flagged"]
plr_sens, plr_spec, plr_ppv, plr_f1, plr_flagged = plr_m["sens"], plr_m["spec"], plr_m["ppv"], plr_m["f1"], plr_m["flagged"]

print(f"  Plaintext Logistic Regression (sklearn): {plain_lr['train_time'] * 1000:.0f}ms")
print(f"  Blind Insight Encrypted LR (k=0, OLS+IRLS): {enc_lr_time_k0:.1f}s")
print(f"  Blind Insight Encrypted LR (k=11, OLS+IRLS): {enc_lr_time:.1f}s")
print(f"    HIPAA suppression: {pair_k11['n_suppressed']} cells replaced (CMS k={CMS_MIN_CELL_SIZE})")

print(f"\nClinical threshold: {CLINICAL_THRESHOLD*100:.2f}% 5-year risk (FDA chemoprevention)")
display(HTML(bc_lr_summary_table(
    enc_train_time=enc_lr_time, plain_train_time=plain_lr["train_time"],
    enc_queries=enc_queries, new_queries=0,
    enc_f1=lr_f1, plain_f1=plr_f1,
    enc_sens=lr_sens, plain_sens=plr_sens,
    enc_spec=lr_spec, plain_spec=plr_spec,
    enc_ppv=lr_ppv, plain_ppv=plr_ppv,
    enc_flagged=lr_flagged, plain_flagged=plr_flagged,
    n_suppressed=pair_k11["n_suppressed"], cms_k=CMS_MIN_CELL_SIZE,
)))

k0_delta = lr0_f1 - lr_f1
print(f"\n  HIPAA k=11 delta: F1 {k0_delta*100:+.1f}pp "
      f"(k=0: {lr0_f1:.3f}, k=11: {lr_f1:.3f}, sklearn: {plr_f1:.3f})")
print(f"  Suppressed {pair_k11['n_suppressed']} of {len(pair_k11['pairwise'])} pairwise cells")

print(f"\nThree models trained on encrypted data - zero records decrypted:")
three_model_rows = build_three_model_rows(
    enc_queries=enc_queries,
    nb_metrics={"f1": nb_f1, "sens": nb_sens, "spec": nb_spec, "ppv": nb_ppv, "flagged": nb_flagged},
    dt_metrics={"f1": dt_f1, "sens": dt_sens, "spec": dt_spec, "ppv": dt_ppv, "flagged": dt_flagged},
    lr_metrics={"f1": lr_f1, "sens": lr_sens, "spec": lr_spec, "ppv": lr_ppv, "flagged": lr_flagged},
)
display(HTML(bc_three_model_comparison_table(three_model_rows)))
print(f"-> All 3 models built from {enc_queries} encrypted aggregate queries, zero records decrypted.")

 ### Real-Time Scoring: Encrypted Records

 Fetch encrypted patient records from Blind Insight, decrypt locally (Data Owner role),
 and score with all three models side-by-side.

 The NB model was trained **entirely on encrypted aggregates** -- it never saw any
 individual record. BCRAT and BCSC use published clinical formulas.

In [ ]:
print("Fetching encrypted + decrypted records from BI...", end=" ", flush=True)
rt = run_bc_realtime_demo(
    client, ORG, DATASET, SCHEMA,
    P_cancer, P_no_cancer, P_enc,
    sample_size=50,
)
print("done")

print(
    f"Query: {rt['enc_query_time'] * 1000:.0f}ms for {rt['rt_count']} records | "
    f"Total: {rt['total_query_time'] * 1000:.0f}ms (encrypted + decrypted)"
)
display(HTML(rt["html"]))

 ### Validate: All Three Models — Encrypted vs sklearn on 10K Test Records

 Each encrypted model is compared against a real-world **sklearn benchmark**
 (NB→NB, Encrypted DT→sklearn CART, Encrypted OLS→sklearn LogisticRegression).
 The gap shows the combined cost of encryption + HIPAA k=11 suppression.

 **Why F1 is low for screening models:**
 With ~3.7% cancer prevalence and a 1.67% threshold, ~38% of patients are flagged
 as high-risk. Even with strong sensitivity, precision (PPV) stays around 6%,
 yielding F1 ≈ 0.11. This is inherent to rare-event screening — BCRAT and BCSC
 show similar patterns. The key metric is **how close** encrypted F1 tracks
 plaintext F1 across all three model types.


In [ ]:
from blind_ml.healthcare import run_bc_full_validation

df_test, _ = load_bc_test_data(TEST_SQLITE_DB)
print(f"  Test set: {len(df_test):,} records")

print("  Validating all three models (BI vs plaintext sklearn)...", end=" ", flush=True)
tv = run_bc_full_validation(
    df_test,
    P_cancer, P_no_cancer, P_enc,
    P_cancer_plain, P_no_cancer_plain, P_plain,
    dt_model=dt,
    plain_dt_model=plain_dt["model"],
    plain_dt_col_names=plain_dt["col_names"],
    lr_beta=lr_model["beta"], lr_dummy_index=lr_model["dummy_index"],
    plain_lr_model=plain_lr["model"],
    plain_lr_col_names=plain_lr["col_names"],
    feature_values=feature_values,
    cohort_prev=cohort_prev,
    pop_rate=POP_5YR_RATE,
    threshold=CLINICAL_THRESHOLD,
    use_sigmoid=True,
)
print(f"done ({tv['pred_time']:.1f}s)")
print(f"  NB agreement: {tv['nb_agreement']*100:.1f}% | DT agreement: {tv['dt_agreement']*100:.1f}% | LR agreement: {tv['lr_agreement']*100:.1f}%")
display(HTML(tv["metrics_html"]))
display(HTML(tv["cm_html"]))

 ### Model Comparison: BI Naive Bayes vs BCRAT vs BCSC

 Now compare the privacy-preserving NB model against two published clinical
 risk calculators at standard clinical thresholds:

 | Threshold | Meaning |
 |-----------|----------|
 | **1.67%** | FDA threshold for chemoprevention eligibility |
 | **3.0%**  | Enhanced screening recommendation (MRI) |

 **Paige et al. (2023)** showed ~20% disagreement between BCRAT, BCSC, and IBIS
 at the individual patient level, even when population-level performance is similar.
 This motivates risk-based screening with multiple models.

In [ ]:
print("  Scoring all test records with NB, BCRAT, and BCSC...", end=" ", flush=True)
comp_start = time.time()
comp = run_model_comparison(df_test, P_cancer, P_no_cancer, P_enc)
comp_time = time.time() - comp_start
print(f"done ({comp_time:.1f}s)")

display(HTML(comp["summary_html"]))
display(HTML(sample_comparison_table(comp["comp_df"], limit=15)))

print(f"""\n--- Key Takeaways ---
1. BCRAT flags {comp['pct_bcrat_167']:.0f}% as high-risk at 1.67% (Paige study: 35.5%)
2. BCSC flags {comp['pct_bcsc_167']:.0f}% as high-risk at 1.67% (Paige study: 20.0%)
3. Models agree on {comp['all_agree']*100:.0f}% of patients (Paige study: ~79%)
4. The BI Naive Bayes model participated in this comparison
   WITHOUT any institution sharing raw patient data.
""")